# LSTM Temperature Model — Kaggle Training Notebook

Notebook này ghi toàn bộ source code vào `/kaggle/working/lstm_temperature/` rồi chạy `main.py` để huấn luyện mô hình LSTM dự báo nhiệt độ.

In [ ]:
import subprocess, os

os.makedirs('/kaggle/working/lstm_temperature', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/data', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/dataset', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/model', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/preprocessing', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/training', exist_ok=True)
os.makedirs('/kaggle/working/lstm_temperature/visualization', exist_ok=True)

print('All directories created.')

In [ ]:
%%writefile /kaggle/working/lstm_temperature/data/__init__.py

# __init__.py for data module

In [ ]:
%%writefile /kaggle/working/lstm_temperature/data/load_data.py
"""
Task: Load raw weather Parquet dataset from Kaggle, filter to the 34 target
      provinces, and return a sorted DataFrame.
"""
import pandas as pd
import numpy as np
import os

_COLS = [
    'latitude', 'longitude', 'valid_time',
    'temperature_celsius', 'apparent_temperature',
    'relative_humidity', 'wind_speed', 'wind_direction',
    'total_precipitation', 'total_cloud_cover',
    'mean_sea_level_pressure', 'surface_pressure',
    'sea_surface_temperature', 'air_density',
]

PROVINCES_COORDS = [
    (21.0, 105.75), (20.75, 106.75), (21.0, 107.25), (21.75, 106.75),
    (22.75, 106.25), (22.5, 104.0), (21.5, 103.0), (21.25, 103.75),
    (21.5, 105.75), (21.25, 105.25), (20.25, 105.75), (20.0, 105.75),
    (18.75, 105.75), (18.25, 105.75), (17.5, 106.5), (16.75, 107.0),
    (16.5, 107.5), (16.0, 108.25), (15.5, 108.5), (15.0, 108.75),
    (13.75, 109.25), (13.0, 109.25), (12.25, 109.25), (14.25, 108.0),
    (14.0, 108.0), (12.5, 108.0), (12.0, 108.5), (11.0, 108.0),
    (11.0, 106.75), (10.75, 106.75), (10.5, 107.25), (10.0, 105.75),
    (10.25, 105.25), (9.25, 105.0)
]



def load_data() -> pd.DataFrame:
    # Auto-detect parquet path under /kaggle/input
    data_path = None
    for root, dirs, files in os.walk("/kaggle/input"):
        # Partitioned parquet directory
        if root.endswith(".parquet") and "weather" in root.lower():
            data_path = root
            break
        # Single parquet file
        for file in files:
            if file.endswith(".parquet") and "weather" in file.lower():
                data_path = os.path.join(root, file)
                break
        if data_path:
            break

    if not data_path:
        raise FileNotFoundError(
            "Không tìm thấy file hoặc thư mục Parquet nào "
            "chứa từ khóa 'weather' trong /kaggle/input!"
        )

    print(f"Đang đọc dữ liệu thời tiết tự động từ đường dẫn: {data_path}")

    unique_lats = list({coord[0] for coord in PROVINCES_COORDS})
    unique_lons = list({coord[1] for coord in PROVINCES_COORDS})

    print("Đang nạp dữ liệu thông minh trực tiếp từ đĩa cứng (RAM tiêu thụ < 200MB)...")
    df = pd.read_parquet(
        data_path,
        columns=_COLS,
        engine='auto',
        filters=[
            ('latitude',  'in', unique_lats),
            ('longitude', 'in', unique_lons),
        ]
    )

    df = df[[c for c in _COLS if c in df.columns]]

    df.set_index(['latitude', 'longitude'], inplace=True)
    df = df.loc[df.index.isin(PROVINCES_COORDS)].reset_index()

    df = df.sort_values(['latitude', 'longitude', 'valid_time']).reset_index(drop=True)

    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')

    print(f"Nạp dữ liệu thành công! Tổng số dòng sau khi lọc: {len(df):,}")
    return df


In [ ]:
%%writefile /kaggle/working/lstm_temperature/dataset/__init__.py

# __init__.py for dataset module

In [ ]:
%%writefile /kaggle/working/lstm_temperature/dataset/sequence_dataset.py
"""
Task: Convert the scaled DataFrame into per-province numpy arrays and build
      lazy-loading sequence indices split temporally (80% train / 20% test)
      with zero geographic boundary overlap.

Memory strategy: store province arrays once; each sequence is a (prov_idx, start)
pointer — no pre-materialised copy of all sequences.
"""
import numpy as np
import torch
from torch.utils.data import Dataset

SEQUENCE_LENGTH = 24   # hours of context → predict T+1


# ── Province array builder ─────────────────────────────────────────────────────

def build_province_arrays(df, feature_cols: list) -> list:
    """Return a list of float32 arrays, one per (lat, lon) province, sorted."""
    arrays = []
    for _, grp in df.groupby(['latitude', 'longitude'], sort=True):
        arr = grp[feature_cols].values.astype(np.float32)
        arrays.append(arr)
    return arrays


# ── Temporal split (per province, no cross-boundary contamination) ─────────────

def build_split_indices(
    province_arrays: list,
    seq_len: int = SEQUENCE_LENGTH,
    train_ratio: float = 0.8,
):
    """
    For each province split at 80% of its time axis.
    Train sequences: x and y both fall inside [0, split).
    Test  sequences: x window starts at or after split → zero leakage.
    """
    train_idx, test_idx = [], []

    for prov_id, arr in enumerate(province_arrays):
        n     = len(arr)
        split = int(n * train_ratio)

        # last valid train start: start + seq_len < split  →  start < split - seq_len
        for i in range(split - seq_len):
            train_idx.append((prov_id, i))

        # first valid test start: x = arr[split : split+seq_len], y = arr[split+seq_len]
        for i in range(split, n - seq_len):
            test_idx.append((prov_id, i))

    return train_idx, test_idx


# ── Dataset ────────────────────────────────────────────────────────────────────

class WeatherSequenceDataset(Dataset):
    """
    Lazy sequence dataset: fetches one (X, y) pair per __getitem__ call
    without pre-materialising the entire sequence tensor in RAM.
    """

    def __init__(
        self,
        province_arrays: list,
        indices: list,
        target_idx: int,
        seq_len: int = SEQUENCE_LENGTH,
    ):
        self.province_arrays = province_arrays
        self.indices         = indices
        self.target_idx      = target_idx
        self.seq_len         = seq_len

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx):
        prov_id, start = self.indices[idx]
        arr = self.province_arrays[prov_id]

        # .copy() required so DataLoader workers get independent buffers
        x = arr[start : start + self.seq_len].copy()
        y = float(arr[start + self.seq_len, self.target_idx])

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.float32),
        )


In [ ]:
%%writefile /kaggle/working/lstm_temperature/model/__init__.py

# __init__.py for model module

In [ ]:
%%writefile /kaggle/working/lstm_temperature/model/lstm_model.py
"""
Task: Define the LSTM model architecture.

Architecture:
  LSTM(input_size, hidden=64, layers=2, dropout=0.1)
  → take last timestep hidden state
  → Linear(64, 32) → ReLU → Dropout(0.1) → Linear(32, 1)
"""
import torch
import torch.nn as nn


class LSTMModel(nn.Module):
    def __init__(
        self,
        input_size:  int,
        hidden_size: int   = 64,
        num_layers:  int   = 2,
        output_size: int   = 1,
        dropout:     float = 0.1,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            # PyTorch applies dropout between LSTM layers (not after the last)
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, input_size)
        out, _ = self.lstm(x)       # (batch, seq_len, hidden)
        out    = out[:, -1, :]      # last timestep → (batch, hidden)
        return self.head(out).squeeze(-1)   # (batch,)


In [ ]:
%%writefile /kaggle/working/lstm_temperature/preprocessing/__init__.py

# __init__.py for preprocessing module

In [ ]:
%%writefile /kaggle/working/lstm_temperature/preprocessing/feature_engineering.py
"""
Task: Add time-encoding, lag, and rolling features; interpolate original NaN values.

Order:
  1. Interpolate raw meteorological NaN (per province group)
  2. Add cyclic time encodings
  3. Add lag features for temperature
  4. Add rolling-mean features for temperature
  5. Drop the tiny residual NaN rows (lag/rolling boundaries)
"""
import numpy as np
import pandas as pd

TARGET_COL = 'temperature_celsius'

_METEO_COLS = [
    'temperature_celsius', 'apparent_temperature',
    'relative_humidity', 'wind_speed', 'wind_direction',
    'total_precipitation', 'total_cloud_cover',
    'mean_sea_level_pressure', 'surface_pressure',
    'sea_surface_temperature', 'air_density',
]


def _interpolate_raw(df: pd.DataFrame) -> pd.DataFrame:
    existing = [c for c in _METEO_COLS if c in df.columns]
    df[existing] = (
        df.groupby(['latitude', 'longitude'], sort=False)[existing]
        .transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    )
    return df


def _add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    dt = df['valid_time']
    df['hour_sin']        = np.sin(2 * np.pi * dt.dt.hour       / 24 ).astype('float32')
    df['hour_cos']        = np.cos(2 * np.pi * dt.dt.hour       / 24 ).astype('float32')
    df['day_of_week_sin'] = np.sin(2 * np.pi * dt.dt.dayofweek  / 7  ).astype('float32')
    df['day_of_week_cos'] = np.cos(2 * np.pi * dt.dt.dayofweek  / 7  ).astype('float32')
    df['month_sin']       = np.sin(2 * np.pi * dt.dt.month      / 12 ).astype('float32')
    df['month_cos']       = np.cos(2 * np.pi * dt.dt.month      / 12 ).astype('float32')
    df['day_of_year_sin'] = np.sin(2 * np.pi * dt.dt.dayofyear  / 365).astype('float32')
    df['day_of_year_cos'] = np.cos(2 * np.pi * dt.dt.dayofyear  / 365).astype('float32')
    return df


def _add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby(['latitude', 'longitude'], sort=False)[TARGET_COL]
    df[f'{TARGET_COL}_lag1'] = grp.shift(1).astype('float32')
    df[f'{TARGET_COL}_lag3'] = grp.shift(3).astype('float32')
    df[f'{TARGET_COL}_lag6'] = grp.shift(6).astype('float32')
    return df


def _add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby(['latitude', 'longitude'], sort=False)[TARGET_COL]
    # shift(1) ensures we never use the current value → no leakage
    df[f'{TARGET_COL}_roll3'] = (
        grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        .astype('float32')
    )
    df[f'{TARGET_COL}_roll6'] = (
        grp.transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
        .astype('float32')
    )
    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = _interpolate_raw(df)
    df = _add_time_features(df)
    df = _add_lag_features(df)
    df = _add_rolling_features(df)
    df = df.dropna().reset_index(drop=True)
    return df


In [ ]:
%%writefile /kaggle/working/lstm_temperature/preprocessing/scaling.py
"""
Task: Fit MinMaxScaler on all numeric feature columns, scale the DataFrame in-place,
      and persist both the scaler and feature column list to disk.
"""
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

SCALER_PATH       = 'scaler_temp.pkl'
FEATURE_COLS_PATH = 'feature_cols_temp.pkl'

_EXCLUDE = {'latitude', 'longitude', 'valid_time'}


def fit_and_scale(df: pd.DataFrame):
    feature_cols = [c for c in df.columns if c not in _EXCLUDE]

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(df[feature_cols].values.astype(np.float32))
    df[feature_cols] = scaled.astype(np.float32)

    with open(SCALER_PATH, 'wb') as f:
        pickle.dump(scaler, f)
    with open(FEATURE_COLS_PATH, 'wb') as f:
        pickle.dump(feature_cols, f)

    print(f"  Scaler          -> {SCALER_PATH}")
    print(f"  Feature columns -> {FEATURE_COLS_PATH}  ({len(feature_cols)} cols)")

    return df, scaler, feature_cols


In [ ]:
%%writefile /kaggle/working/lstm_temperature/training/__init__.py

# __init__.py for training module

In [ ]:
%%writefile /kaggle/working/lstm_temperature/training/trainer.py
"""
Task: Run the training loop — one epoch of gradient updates on train set,
      followed by inference on val set, then log MSE / RMSE / MAE for both.
"""
import numpy as np
import torch


# ── Metric helpers ─────────────────────────────────────────────────────────────

def _metrics(preds: np.ndarray, targets: np.ndarray) -> tuple[float, float, float]:
    mse  = float(np.mean((preds - targets) ** 2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(preds - targets)))
    return mse, rmse, mae


# ── Single epoch pass ──────────────────────────────────────────────────────────

def _run_epoch(model, loader, optimizer, criterion, device, train: bool):
    model.train(train)
    preds_buf, tgts_buf = [], []

    with torch.set_grad_enabled(train):
        for X_b, y_b in loader:
            X_b = X_b.to(device, non_blocking=True)
            y_b = y_b.to(device, non_blocking=True)

            out  = model(X_b)
            loss = criterion(out, y_b)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            preds_buf.append(out.detach().cpu().numpy())
            tgts_buf.append(y_b.detach().cpu().numpy())

    preds   = np.concatenate(preds_buf)
    targets = np.concatenate(tgts_buf)
    return _metrics(preds, targets)


# ── Full training loop ─────────────────────────────────────────────────────────

def train_model(model, train_loader, val_loader, optimizer, criterion, device, epochs: int = 15) -> dict:
    history = {k: [] for k in [
        'epoch',
        'train_mse', 'train_rmse', 'train_mae',
        'val_mse',   'val_rmse',   'val_mae',
    ]}

    for epoch in range(1, epochs + 1):
        tr_mse, tr_rmse, tr_mae = _run_epoch(
            model, train_loader, optimizer, criterion, device, train=True
        )
        vl_mse, vl_rmse, vl_mae = _run_epoch(
            model, val_loader, None, criterion, device, train=False
        )

        for key, val in zip(history.keys(), [
            epoch,
            tr_mse, tr_rmse, tr_mae,
            vl_mse, vl_rmse, vl_mae,
        ]):
            history[key].append(val)

        print(
            f"Epoch [{epoch:02d}/{epochs}]  "
            f"Train MSE: {tr_mse:.4f}, Val MSE: {vl_mse:.4f}  |  "
            f"Train RMSE: {tr_rmse:.4f}, Val RMSE: {vl_rmse:.4f}  |  "
            f"Train MAE: {tr_mae:.4f}, Val MAE: {vl_mae:.4f}"
        )

    return history


In [ ]:
%%writefile /kaggle/working/lstm_temperature/training/evaluator.py
"""
Task: Evaluate the trained model on the held-out test Dataset and
      print a formatted summary table of Test MSE / RMSE / MAE.
"""
import numpy as np
import torch
from torch.utils.data import DataLoader


def evaluate_test(model, test_dataset, device, batch_size: int = 1024) -> tuple[float, float, float]:
    loader = DataLoader(
        test_dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'),
    )

    model.eval()
    preds_buf, tgts_buf = [], []

    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(device, non_blocking=True)
            out = model(X_b)
            preds_buf.append(out.cpu().numpy())
            tgts_buf.append(y_b.numpy())

    preds   = np.concatenate(preds_buf)
    targets = np.concatenate(tgts_buf)

    mse  = float(np.mean((preds - targets) ** 2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(preds - targets)))

    w = 54
    print()
    print("=" * w)
    print(f"{'TEST SET EVALUATION RESULTS':^{w}}")
    print("=" * w)
    print(f"  {'Metric':<22} {'Value':>16}")
    print("-" * w)
    print(f"  {'Test MSE':<22} {mse:>16.6f}")
    print(f"  {'Test RMSE (deg C)':<22} {rmse:>16.6f}")
    print(f"  {'Test MAE  (deg C)':<22} {mae:>16.6f}")
    print("=" * w)

    return mse, rmse, mae


In [ ]:
%%writefile /kaggle/working/lstm_temperature/visualization/__init__.py

# __init__.py for visualization modules

In [ ]:
%%writefile /kaggle/working/lstm_temperature/visualization/plot_loss.py
"""
Task: (1) Plot Train vs Val MSE loss curve and save as 'loss_curve.png'.
      (2) Save the full per-epoch history (MSE, RMSE, MAE) to 'training_history.csv'.
"""
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe for Kaggle notebooks
import matplotlib.pyplot as plt

LOSS_CURVE_PATH = 'loss_curve.png'
HISTORY_CSV_PATH = 'training_history.csv'


def plot_loss_curve(history: dict, save_path: str = LOSS_CURVE_PATH) -> None:
    epochs = history['epoch']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, history['train_mse'], 'o-',  label='Train MSE', linewidth=2)
    ax.plot(epochs, history['val_mse'],   's--', label='Val MSE',   linewidth=2)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('MSE Loss', fontsize=12)
    ax.set_title('LSTM Temperature Model — Train vs Val MSE Loss', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(epochs)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Loss curve      -> {save_path}")


def save_history(history: dict, save_path: str = HISTORY_CSV_PATH) -> None:
    pd.DataFrame(history).to_csv(save_path, index=False)
    print(f"  Training history -> {save_path}")


In [ ]:
%%writefile /kaggle/working/lstm_temperature/main.py
"""
Task: Orchestrate the full pipeline — load → engineer → scale → build sequences
      → train LSTM → evaluate → export artifacts.

Run on Kaggle:
    !python /kaggle/working/lstm_temperature/main.py
"""
import os
import sys

# Allow sibling-package imports regardless of CWD
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from data.load_data import load_data
from preprocessing.feature_engineering import engineer_features
from preprocessing.scaling import fit_and_scale
from dataset.sequence_dataset import (
    build_province_arrays,
    build_split_indices,
    WeatherSequenceDataset,
    SEQUENCE_LENGTH,
)
from model.lstm_model import LSTMModel
from training.trainer import train_model
from training.evaluator import evaluate_test
from visualization.plot_loss import plot_loss_curve, save_history

# ── Hyper-parameters ───────────────────────────────────────────────────────────
EPOCHS      = 15
BATCH_SIZE  = 1024
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.1
LR          = 0.001
TARGET_COL  = 'temperature_celsius'
MODEL_PATH  = 'lstm_weather_model_temp.pt'


def main() -> None:
    # ── Device ──────────────────────────────────────────────────────────────────
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device : {device}")
    if device.type == 'cuda':
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
    use_pin = device.type == 'cuda'

    # ── 1. Load ─────────────────────────────────────────────────────────────────
    print("\n[1/6] Loading data ...")
    df = load_data()
    n_locs = df['latitude'].nunique()
    print(f"  {len(df):>10,} rows  |  {n_locs} unique locations")

    # ── 2. Feature engineering ───────────────────────────────────────────────────
    print("\n[2/6] Engineering features ...")
    df = engineer_features(df)
    print(f"  {len(df):>10,} rows  |  {len(df.columns)} columns")

    # ── 3. Scale ─────────────────────────────────────────────────────────────────
    print("\n[3/6] Scaling ...")
    df, _, feature_cols = fit_and_scale(df)
    target_idx = feature_cols.index(TARGET_COL)

    # ── 4. Sequences ─────────────────────────────────────────────────────────────
    print("\n[4/6] Building sequences ...")
    province_arrays = build_province_arrays(df, feature_cols)
    del df   # release ~180 MB

    train_idx, test_idx = build_split_indices(province_arrays, SEQUENCE_LENGTH)
    print(f"  Train: {len(train_idx):,}  |  Test: {len(test_idx):,}  "
          f"(seq_len={SEQUENCE_LENGTH})")

    train_ds = WeatherSequenceDataset(province_arrays, train_idx, target_idx, SEQUENCE_LENGTH)
    test_ds  = WeatherSequenceDataset(province_arrays, test_idx,  target_idx, SEQUENCE_LENGTH)

    # The val_loader re-uses the test split for epoch-level monitoring
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=use_pin, persistent_workers=True,
    )
    val_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=use_pin, persistent_workers=True,
    )

    # ── 5. Model ─────────────────────────────────────────────────────────────────
    input_size = len(feature_cols)
    model      = LSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS, 1, DROPOUT).to(device)
    optimizer  = optim.Adam(model.parameters(), lr=LR)
    criterion  = nn.MSELoss()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n[5/6] Model ready  |  input_size={input_size}  |  params={n_params:,}")

    # ── 6. Train ─────────────────────────────────────────────────────────────────
    print(f"\n[6/6] Training  ({EPOCHS} epochs, batch={BATCH_SIZE}) ...")
    print("-" * 95)
    history = train_model(
        model, train_loader, val_loader, optimizer, criterion, device, EPOCHS
    )
    print("-" * 95)

    # ── Evaluate on independent test set ────────────────────────────────────────
    evaluate_test(model, test_ds, device, BATCH_SIZE)

    # ── Export artifacts ─────────────────────────────────────────────────────────
    print("\n[Export]")
    plot_loss_curve(history)
    save_history(history)

    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'input_size':       input_size,
            'hidden_size':      HIDDEN_SIZE,
            'num_layers':       NUM_LAYERS,
            'output_size':      1,
            'dropout':          DROPOUT,
            'sequence_length':  SEQUENCE_LENGTH,
            'target_cols':      [TARGET_COL],
            'scaler_name':      'scaler_temp.pkl',
            'feature_cols_name':'feature_cols_temp.pkl',
        },
        MODEL_PATH,
    )
    print(f"  Checkpoint      -> {MODEL_PATH}")
    print("\n[Done]")


if __name__ == '__main__':
    main()


In [ ]:
!python /kaggle/working/lstm_temperature/main.py